# vgra-yolo: Part A evaluation session (I1 + I3 + I4 + I2)

One notebook for the evaluation of every trained detector (docs/PROTOCOL.md, Part A).

**Two modes, set in the first code cell:**
- `OPEN_TEST = False` (**dry run**, default): validation split only. Checks the whole pipeline on Kaggle
  with the detectors that exist. Safe to run any time; the test split is never read.
- `OPEN_TEST = True` (**the one test session**): only when every detector in `RUNS` has FINISHED and the code
  is tagged. Runs predictions on valid and test (I1), the RQ1 premise test on the test pairs (I3), standard
  Ultralytics validation on test (I4) and the patient-level analysis (I2). The test split is opened once:
  if a bug is found afterwards, it is fixed, everything is rerun, and the event is logged (protocol section 2).

Settings: GPU **T4 x1** is enough (about 1-1.5 h for 8 detectors), Internet **on**.
Inputs: the images, `vgra-index`, and every dataset or notebook output that holds the trained runs
(`runs/<name>/weights/best.pt`): the B0_s0 + B1_s0 output, `vgra-b0-s1-s2`, `vgra-v8-s0-s1`, `vgra-v8r-s0-s1`.
The output of this notebook (`/kaggle/working/runs/...`) holds every table; download it or make it a dataset.

In [ ]:
# Settings
REPO = "https://github.com/nithoukhan1/vgra-yolo.git"
REF = "main"        # dry run: main is fine; the test session: a fixed tag made just before (e.g. "parta-test-v1")
OPEN_TEST = False   # True ONLY for the one test session
RUNS = ["B0_s0", "B1_s0"]  # test session: ["B0_s0", "B0_s1", "B0_s2", "B1_s0", "V8_s0", "V8_s1", "V8R_s0", "V8R_s1"]

In [ ]:
# Clone the fixed code, install, check
%cd /kaggle/working
!rm -rf vgra-yolo
!git clone --quiet {REPO} vgra-yolo
%cd /kaggle/working/vgra-yolo
!git checkout --quiet {REF}
!git log -1 --oneline
!pip install -q -r requirements.txt
!pip install -q -e .
!python scripts/00_check_setup.py

In [ ]:
# Every run must be found and FINISHED before anything is evaluated
import sys
sys.path.insert(0, "/kaggle/working/vgra-yolo")
from vgra.runs import find_previous_run, is_finished, read_metrics

problems = []
for r in RUNS:
    p = find_previous_run(r, "/kaggle/input")
    if p is None or not (p / "weights" / "best.pt").is_file():
        problems.append(f"{r}: not found under /kaggle/input (attach its dataset)")
        continue
    v = read_metrics(p).get("valid", {})
    print(f"{r}: {p} | finished {is_finished(p)} | valid mAP50 {v.get('map50')} | mAP50-95 {v.get('map50_95')}")
    if not is_finished(p):
        problems.append(f"{r}: not finished")
if problems:
    raise SystemExit("STOP: " + "; ".join(problems))
if OPEN_TEST:
    print("\nOPEN_TEST is True: this session opens the test split (protocol section 2).")
else:
    print("\nDry run: validation split only.")
RUN_ARGS = " ".join(RUNS)

In [ ]:
# I1: predictions on valid (every run), then on test (test session only)
!python scripts/10_predict.py --runs {RUN_ARGS} --split valid
if OPEN_TEST:
    !python scripts/10_predict.py --runs {RUN_ARGS} --split test --open-test

In [ ]:
# I3: RQ1 premise test on the test pairs (test session only; on valid it was Gate 2, already done)
if OPEN_TEST:
    !python scripts/04_premise_test.py --runs {RUN_ARGS} --split test --open-test
else:
    !python scripts/04_premise_test.py --runs {RUN_ARGS} --split valid --limit 64 --boot 50 --out /kaggle/working/dry

In [ ]:
# I4: standard Ultralytics validation (detector table T2)
SPLIT = "test --open-test" if OPEN_TEST else "valid"
!python scripts/11_val_detectors.py --runs {RUN_ARGS} --split {SPLIT}

In [ ]:
# I2: patient-level analysis (CPU), thresholds from valid
EVAL = "test" if OPEN_TEST else "valid"
!python scripts/12_study_level.py --pred /kaggle/working/runs/predictions --runs {RUN_ARGS} --eval {EVAL}

In [ ]:
# What was produced
!find /kaggle/working/runs -maxdepth 2 \( -name "*.csv" -o -name "*.json" -o -name "summary.txt" \) | sort